# 使用@tool装饰器定义工具

## 1、自定义工具的描述description

举例1：

函数使用@tool装饰器修饰以后就可以被模型识别使用

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool
from rich import print as rprint

@tool
def get_weather(city: str):
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

补充docstring

In [7]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool
from rich import print as rprint

@tool
def get_weather(city: str):
    '''天气查询工具'''
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '天气查询工具',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

添加工具描述

In [8]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool
from rich import print as rprint

@tool(description='获取具体城市的天气情况')
def get_weather(city: str):
    '''天气查询工具'''
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取具体城市的天气情况',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

## 2、更改工具名称

默认情况，使用函数名作为工具名称，

举例

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool
from rich import print as rprint

@tool(name_or_callable="getWeather")
def get(city: str):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"

print(convert_to_openai_tool(get))

{'type': 'function', 'function': {'name': 'getWeather', 'description': '天气查询工具', 'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}}}


## 3、自定义args_schema

### 3.1 使用pydantic模型定义
举例1


In [13]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool
from pydantic import BaseModel
from rich import print as rprint


class WeatherInput(BaseModel):
    city: str


@tool(args_schema=WeatherInput)
def getweather(city: str):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(getweather))

{
    'type': 'function',
    'function': {
        'name': 'getweather',
        'description': '天气查询工具',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

举例2

In [18]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool
from pydantic import BaseModel,Field
from rich import print as rprint


class WeatherInput(BaseModel):
    city: str = Field(
        description='具体的城市',
    )


@tool(args_schema=WeatherInput)
def getweather(city: str):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(getweather))

{
    'type': 'function',
    'function': {
        'name': 'getweather',
        'description': '天气查询工具',
        'parameters': {
            'properties': {'city': {'description': '具体的城市', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

举例三：

In [29]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool
from pydantic import BaseModel,Field
from rich import print as rprint
from typing import Literal


class WeatherInput(BaseModel):
    city: str = Field(
        description='具体的城市',)

    unit: (Literal['celsius','fahrenheit']
    )


@tool(args_schema=WeatherInput)
def getweather(city: str,unit : str = 'celsius'):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(getweather))

{
    'type': 'function',
    'function': {
        'name': 'getweather',
        'description': '天气查询工具',
        'parameters': {
            'properties': {
                'city': {'description': '具体的城市', 'type': 'string'},
                'unit': {'enum': ['celsius', 'fahrenheit'], 'type': 'string'}
            },
            'required': ['city', 'unit'],
            'type': 'object'
        }
    }
}

### 3.2 方式二：json schema定义

In [30]:

json_schema = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "获取当日天气，可选未来五日天气预报",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {"type": "string"},
                "units": {"type": "string"},
                "include_forecast": {"type": "boolean"}
            },
            "required": ["location", "units", "include_forecast"]
        },
    }
}

@tool(args_schema=json_schema)
def getweather(city: str,unit : str = 'celsius'):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(getweather))

{
    'type': 'function',
    'function': {
        'name': 'getweather',
        'description': '天气查询工具',
        'parameters': {
            'type': 'function',
            'function': {
                'name': 'get_weather',
                'description': '获取当日天气，可选未来五日天气预报',
                'parameters': {
                    'type': 'object',
                    'properties': {
                        'location': {'type': 'string'},
                        'units': {'type': 'string'},
                        'include_forecast': {'type': 'boolean'}
                    },
                    'required': ['location', 'units', 'include_forecast']
                }
            }
        }
    }
}